In [1]:
import pandas as pd
import re
import numpy as np
import ast
import json

In [ ]:
# initialize dataframes for combined data and keywords
df = pd.read_excel('all_data-2026_02_11_161834.xlsx') # combined raw data
kw = pd.read_csv('ubpp_grid_keywords_upadated.csv') # keywords

In [3]:
# print shape and column names for combined data and keywords
print(df.shape)
print(df.columns)
print(kw.shape)
print(kw.columns)

(94592, 18)
Index(['source_project_id', 'name', 'funding_amount', 'country_raw',
       'start_date', 'recipient_name', 'sector', 'source_data_set',
       'funder_name', 'funder_type', 'instrument', 'other_columns', 'goal_raw',
       'phase_raw', 'end_date', 'id', 'description', 'country_standardized'],
      dtype='str')
(186, 6)
Index(['keyword', 'concept_family', 'series', 'strategy_level', 'goal',
       'additional_description'],
      dtype='str')


In [4]:
# lists for valid entries in ubbp columns
# is currently not used

concept_family_list = [
    #oceans
    "mpas_area_based", "highseas_governance", "fisheries_management", "ocean_tourism", 
    "ocean_livelihoods_enterprises", "ocean_governance",
    #food
    "food_systems_planning", "biosecurity_food_safety", "landscapes_restoration_production", 
    "food_tourism_culture", "food_logistics_markets", "food_community_nutrition", "food_governance", 
    #finance
    "finance_system_design_regulation", "finance_capacity_building", "finance_enterprise_support", 
    "finance_instruments_vehicles"]

series_list = [
    "protection", "sustainable use", "governance", "sustainable production", "optimal processing distribution and consumption", 
    "policy & governance", "systems & architecture", "finance delivery & instruments"]

strategy_level_list = [
    "regional", "sub-regional", "national", "sub-national", "local", "foundational"]

goal_list = ["oceans", "food", "finance"]

In [5]:
# creating the combined text field, same code from v3

keep_keys = [ #for cleaning other_columns for embeddings
    # can cross-reference these in the field_mapping.xlsx doc OR in the init cleaning script
    '2050_ip_outcome',
    '2050_theme',
    'agencies',
    'department',
    'desc',
    'description',
    'description_narrative',
    'details_page',
    'focal_areas',
    'purpose_name',
    'type',
    'sector',
    'sector_other',
    'short_desc',
    'subsector',
    'tags',
    'theme',
    'implementing_agency',
    'partner_agency',
    'primary_subject',
    'program',
    'project_developent_objective',
    'support_strategies'
]

def clean_other_columns(df, col="other_columns", keep_keys=None, new_col="other_columns_cleaned"):
    if keep_keys is None:
        raise ValueError("keep_keys list must be provided")

    def safe_parse(val):
        """Handle Python-style dict strings with NaN, None, etc."""
        if pd.isna(val):
            return {}
        if isinstance(val, dict):
            return val
        if not isinstance(val, str):
            return {}

        # Fix unquoted nan / None / bools
        val_fixed = (
            val.replace("nan", "None")
               .replace("None", "null")  # temporarily JSONify
               .replace("True", "true")
               .replace("False", "false")
        )

        # Try AST first (Python-style)
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

        # Try JSON as fallback
        try:
            return json.loads(val_fixed.replace("'", '"'))
        except Exception:
            return {}

    def extract_keys(val):
        parsed = safe_parse(val)
        kept = [
            str(parsed.get(k, "")) for k in keep_keys
            if k in parsed and pd.notna(parsed[k]) and parsed[k] != ""
        ]
        return " ".join(kept).strip()

    df[new_col] = df[col].apply(extract_keys)
    return df

# run it
df = clean_other_columns(df=df,keep_keys=keep_keys)

# combined text field should include title, sector (where it exists), 
# and other_columns after cleaning
df['final_combined_text'] = (
    df['other_columns_cleaned'].astype(str) + ' | ' 
    +  df['name'].astype(str) + ' | ' 
    +  df['sector'].astype(str)
)

In [6]:
# setup regex column for keywords

def format_keyword(str):
    str = str.rstrip()
    str = str.replace("(", "")
    str = str.replace(")", "")
    str = str.replace(" &", "&?")
    str = str.replace("-", "-?")
    return str

# creates regex for each keyword
# possibly unnessary regexes are included for broader coverage, false positives are unlikely
def convert_keyword_to_regex(rec):
    kw_str = format_keyword(rec['keyword'])
    kw_str1 = kw_str[:-1]                   # remove last char
    kw_str2 = kw_str + r"s"                 # concatenate 's', for possible plural noun
    pipe = r"|"
    return r"\b(" + kw_str + pipe + kw_str1 + pipe + kw_str2 + r")\b"

In [7]:
# create and populate regex column

kw = kw.drop_duplicates(subset=['keyword'])
kw['regex'] = kw.apply(convert_keyword_to_regex, 1)

In [8]:
# removes duplicate substrings based on delim
def remove_duplicates(str, delim=", "):
    split_str = str.split(delim)
    unique_split_str = dict.fromkeys(split_str)
    result = delim.join(unique_split_str)
    return result

# returns a Series containing every row where val matches kw[regex]
def match_keyword_records(kw, val=""):
    try:
        kw_recs = kw[kw['regex'].apply(lambda x: bool(re.search(x, val)))]
        return kw_recs
    except (AttributeError, TypeError, IndexError) as e:
        return None

# returns a comma separated string containing all values in col
# values are not duplicated
def get_keyword_column(kw_recs, col=''):
    if kw_recs is None:
        return ""
    try:
        col_str = kw_recs[col].str.cat(sep=", ")
        col_str_cleaned = remove_duplicates(col_str, delim=", ")
        return col_str_cleaned
    except (TypeError, KeyError, IndexError) as e:
        return ""

In [9]:
def get_concept_family(kw_recs):
    return get_keyword_column(kw_recs, col='concept_family')

def get_series(kw_recs):
    return get_keyword_column(kw_recs, col='series')
    
def get_strategy_level(kw_recs):
    return get_keyword_column(kw_recs, col='strategy_level')
    
def get_goal(kw_recs):
    return get_keyword_column(kw_recs, col='goal')

# adds ubbp columns to rec from matching keywords
def add_ubbp_columns(rec):
    val = rec['other_columns']
    kw_recs = match_keyword_records(kw, val)
    rec['ubbp_grid_keywords'] = get_keyword_column(kw_recs, 'keyword')
    rec['ubbp_concept_family'] = get_keyword_column(kw_recs, 'concept_family')
    rec['ubbp_series'] = get_keyword_column(kw_recs, 'series')
    rec['ubbp_strategy_level'] = get_keyword_column(kw_recs, 'strategy_level')
    rec['ubbp_goal'] = get_keyword_column(kw_recs, 'goal')
    return rec

# testing
# rec = df.iloc[0]
# kw_recs = match_keyword_records(kw=kw, val=rec['other_columns'])
# count = 0

# for i in range(len(df)):
#     rec = df.iloc[i]
#     kw_recs = match_keyword_records(kw=kw, val=rec['other_columns'])
#     if (get_keyword_column(kw_recs=kw_recs, col='keyword')) != "":
#         print(f"Record {i}: " + rec['other_columns'])
#         print("Keywords: " + get_keyword_column(kw_recs=kw_recs, col='goal'))
#         count += 1
# print(f"Final count: ${count}")

In [10]:
# add ubbp columns to combined data
df = df.apply(add_ubbp_columns, 1)

In [11]:
# export
output_file_name = "labelled_data.xlsx"
df.to_excel(output_file_name, index=False)